# Allsky cloud detection: one case

A short run of frames of one night through `allsky_lib.py`: `SkyAnalyzer` (the whole-sky indicators and the cloud maps of
each frame) and `StateTracker` (the state in time and the veil), with the drawing of `allsky_regions.py`. Method of 2026-10-29; the report
(`allsky_report.py` → `~/data/allsky/report/allsky_report.html`) describes it.

**Stars.** Which stars a clear sky would show is decided from the frame itself: its own zero point (bright, unsaturated
stars), against the clear-sky noise of the frame (lower quartile of the local noise), without the stars and places the
clear-sky calibration (`allsky_calib.py calibrate <clear nights>`, done by hand after cleaning or when dirt is suspected)
finds unreliable. *Expected* = predicted signal above 1.5 × the detection threshold, *sure* = above 3 ×.
Maps smoothed over a few degrees (the width follows the number of expected stars):

- **possible** cloud: more than half of the expected stars missing, or the seen stars dimmed by more than 0.4 mag;
- **sure** cloud: (almost) none of the sure stars seen — an opaque cloud lets no star through;
- **cirrus**: possible but not sure — the stars still shine through (kept as a label for the plots, drawn as possible).

A veil over the whole sky lowers the frame's own zero point and is not seen by the map; `transp` (zero point against
`zp_clear`) sees it: when its smoothed value falls below the thin limit while the stars are still seen everywhere (smoothed
`star_seen` ≥ 0.8), the **whole sky is marked yellow** (`veil`).

**Difference** — regions of the smoothed difference with the previous frame after derotating the sky, in units of the
pixel noise: possible = seeds above 3 grown down to 1.5, sure = seeds above 6 grown down to 3.

Panels:

- **left**: the frame (alt ≥ 30°) with both methods together, as for the observers (two marks only) — red outline: cloud
  that hides everything (star map sure, or difference sure only where the star map cannot tell: near the Moon, too few
  stars; a strong difference where the stars shine through is yellow), light yellow fill: cirrus, semi-transparent or
  uncertain cloud (possible, either method); light cyan area: the Moon mask, no detection inside;
- **middle**: the raw difference (derotated, no smoothing, ±6 pixel noises; red = brighter than the previous frame,
  blue = darker) with the difference detections — yellow outline possible, black sure, light cyan its Moon mask;
- **right of it**: the expected stars (green seen, red missing; large dots = sure stars) with the star detections —
  yellow outline possible, red sure, light cyan the star map's Moon mask (the glare + 10°);
- **plots**: the whole-sky indicators (`transp`, `star_seen` of the whole sky and of the zenith cap, `diff_ls_n`,
  smoothed as in the state), the state band on top, and the sky share of each detection — the observers' marks (yellow,
  red) and each method alone (blue stars, orange difference); cursor at the current frame.
  Dashed: `STATE_LIMITS` (the thin limit of `transp` moves down with a bright Moon) and `VEIL_SEEN`.

Pick a case in `CASE` (or add one to `CASES`); `"live"` takes the newest frames of tonight. Reads (`N` + 1) frames for
`STEP` = 1, 2 `N` otherwise (each shown frame needs the frame just before it for the difference); ~1.5 s per frame.

In [10]:
# configuration

# name: (UT date, start "HHMM", number of frames N, STEP = every STEP-th frame, what to look at);
# UT date None = today's directory, start "last" = the run ends with the newest frame
CASES = {
    "live":             (None, "last", 14, 1, "the newest frames of tonight"),
    "moonlit_1029":     ("20261029", "0530", 14, 2, "clear by the telescopes, bright Moon: the first star map called it cloudy next to the Moon"),
    "forming_1005":     ("20261005", "0130", 14, 2, "cloud forming in place, no Moon: the star map sees it in the S from 01:40, the difference hardly"),
    "veil_1007":        ("20261007", "0215", 14, 3, "thin still veil, no Moon: both telescopes lost up to 1.5 mag 02:20-03:40; transp sees it, the maps do not"),
    "clouds_1025":      ("20261025", "0300", 14, 2, "clouds coming in under a full Moon: diff_ls_n rises ~40 min before the weather_overwatch alert (03:54)"),
    "clouds_1027":      ("20261027", "0545", 14, 1, "clouds arrive after a clearing, full Moon: alert 06:16, telescope points bad 06:10-06:25"),
    "altocumulus_0928": ("20260928", "0510", 10, 1, "moonlit altocumulus over half of the sky: which parts are cloudy"),
    "cirrus_1018":      ("20261018", "0410", 10, 1, "cirrus streaks, low Moon: differences see them, stars hardly"),
    "clouding_1010":    ("20261010", "0545", 14, 2, "clear sky clouding over, no Moon"),
    "clear_1017":       ("20261017", "0400", 10, 2, "clear reference, no Moon"),
    # small difference regions on an otherwise clear sky (found in the stored region maps): is the region the cloud, or larger?
    "isolated_1005":    ("20261005", "0735", 12, 1, "small difference regions on a clear sky, no Moon"),
    "isolated_1025a":   ("20261025", "0305", 18, 1, "small difference regions on a clear sky, full Moon"),
    "isolated_1025b":   ("20261025", "0555", 14, 1, "small difference regions between cloud episodes, full Moon"),
    "isolated_1027":    ("20261027", "0510", 10, 1, "small difference regions on a clear sky before the clouds, bright Moon"),
}
CASE = "altocumulus_0928"

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from allsky_lib import *
from allsky_regions import RegionAnalyzer, draw, draw_diff, draw_stars, det_areas

plt.rcParams["animation.embed_limit"] = 80
day, start, n_frames, step, about = CASES[CASE]
if day is None:
    day = Time.now().strftime("%Y%m%d")

ra = RegionAnalyzer()
if ra.an.calib is None:
    print(f"no clear-sky calibration ({CALIB_NPZ}): all stars used; run allsky_calib.py calibrate <clear nights>")
paths = day_frames(day)
i0 = max(len(paths) - 1 - step * (n_frames - 1), 1) if start == "last" else frame_index(paths, start)
shown = [i0 + step * k for k in range(n_frames) if i0 + step * k < len(paths)]
print(f"{CASE}: {about}\n{day}: {os.path.basename(paths[shown[0]])} - {os.path.basename(paths[shown[-1]])}, {len(shown)} frames, every {step}")

frames = [ra.frame(paths[k], paths[k - 1]) for k in shown]
table = pd.DataFrame([{**f["row"], **det_areas(f["det"], ra.g)} for f in frames])
table["t"] = pd.to_datetime(table.frame.str[:13], format="%Y%m%d_%H%M")
table[["frame", "state", "transp_s", "star_seen", "zen_star_seen", "diff_s", "star_possible", "star_sure", "star_cirrus", "diff_possible", "diff_sure", "veil", "moon_alt", "twilight"]].round(2)

altocumulus_0928: moonlit altocumulus over half of the sky: which parts are cloudy
20260928: 20260928_0510.jpg - 20260928_0532.jpg, 10 frames, every 1


KeyError: "['veil'] not in index"

## The run

In [ ]:
STATE_COLORS = {"clear": "#4caf50", "thin": "#ffc107", "cloudy": "#ff7043", "overcast": "#757575"}

fig = plt.figure(figsize=(22, 7.4), dpi=60)
gs = fig.add_gridspec(5, 4, width_ratios=[1, 1, 1, 1.15], height_ratios=[0.3, 1, 1, 1, 1])
ax_img = fig.add_subplot(gs[:, 0])
ax_diff = fig.add_subplot(gs[:, 1])
ax_star = fig.add_subplot(gs[:, 2])
ax_band = fig.add_subplot(gs[0, 3])
ax_tr = fig.add_subplot(gs[1, 3], sharex=ax_band)
ax_sf = fig.add_subplot(gs[2, 3], sharex=ax_band)
ax_df = fig.add_subplot(gs[3, 3], sharex=ax_band)
ax_ar = fig.add_subplot(gs[4, 3], sharex=ax_band)

ends = list(table.t.iloc[1:]) + [table.t.iloc[-1] + table.t.diff().median()]
for (_, row), end in zip(table.iterrows(), ends):
    ax_band.axvspan(row.t, end, color=STATE_COLORS[row.state], alpha=0.4 if row.twilight else 1, lw=0)
ax_band.set_yticks([])
ax_band.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in STATE_COLORS.values()], labels=list(STATE_COLORS), ncol=4, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False)
ax_tr.plot(table.t, table.transp, ".", color="lightgray", ms=4)
ax_tr.plot(table.t, table.transp_s.fillna(-2.4), "k.-", ms=4)
ax_tr.plot(table.t, STATE_LIMITS["thin_transp"] + STATE_LIMITS["thin_transp_moon"] * table.moon_x, color="gray", ls="--", lw=0.8)
for key in ("cloudy_transp", "overcast_transp"):
    ax_tr.axhline(STATE_LIMITS[key], color="gray", ls="--", lw=0.8)
ax_tr.set_ylabel("transp [mag]")
ax_sf.plot(table.t, table.star_seen, "k.-", ms=4, label="whole sky")
ax_sf.plot(table.t, table.zen_star_seen, ".-", color="tab:blue", ms=4, label="zenith cap")
ax_sf.axhline(VEIL_SEEN, color="gray", ls="--", lw=0.8)   # a veil needs at least this share seen
ax_sf.set_ylim(-0.05, 1.05)
ax_sf.set_ylabel("star_seen")
ax_sf.legend(fontsize=7, loc="lower left")
ax_df.plot(table.t, table.diff_s, "k.-", ms=4)
for key in ("thin_diff", "cloudy_diff"):
    ax_df.axhline(STATE_LIMITS[key], color="gray", ls="--", lw=0.8)
ax_df.set_yscale("log")
ax_df.set_ylabel("diff_ls_n")
ax_ar.fill_between(table.t, table.possible, color="yellow", alpha=0.35, lw=0, label="marked yellow or red")
ax_ar.plot(table.t, table.sure, color="red", lw=2, label="marked red")
for key, color, ls, label in (("star_possible", "tab:blue", "--", "stars possible"), ("star_sure", "tab:blue", "-", "stars sure"), ("star_cirrus", "c", ":", "stars cirrus"),
                              ("diff_possible", "tab:orange", "--", "difference possible"), ("diff_sure", "tab:orange", "-", "difference sure")):
    ax_ar.plot(table.t, table[key], color=color, ls=ls, label=label)
ax_ar.set_ylim(-0.03, 1.03)
ax_ar.set_ylabel("detections\n(share of sky)")
ax_ar.legend(fontsize=7, ncol=4, loc="upper left")
cursors = [ax.axvline(table.t.iloc[0], color="tab:blue", lw=1.5) for ax in (ax_band, ax_tr, ax_sf, ax_df, ax_ar)]
for ax in (ax_tr, ax_sf, ax_df, ax_ar):
    ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()


def update(n):
    f, row = frames[n], table.iloc[n]
    for ax in (ax_img, ax_diff, ax_star):
        ax.clear()
    draw(ax_img, f["img"], f["det"], ra.sky, f"{row.frame}  {row.state.upper()}{' (twilight)' if row.twilight else ''}\n"
         f"transp {row.transp_s:.2f}  star_seen {row.star_seen:.2f}  diff_ls_n {row.diff_s:.2f}  Moon {row.moon_alt:.0f}°", f["moon_diff"])
    draw_diff(ax_diff, f["diff"], f["noise"], f["det"], ra.sky, "difference (red brighter, blue darker; ±6 pixel noises)", f["moon_diff"])
    draw_stars(ax_star, f["stars"], f["det"], ra.sky, f"expected stars ({row.n_exp}, map sigma {row.sigma:.1f}°)", f["moon_stars"])
    for c in cursors:
        c.set_xdata([row.t, row.t])
    return []


anim = FuncAnimation(fig, update, frames=len(table), interval=900)
plt.close(fig)
HTML(anim.to_jshtml())